In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from collections import defaultdict
from data_preprocess import encode_features, construct_df, PCA, FeatureSelection

In [4]:
class NaiveBayesClassifier:
    def __init__(self, data=None):
        self.edidble_counts = 0
        self.poisonous_counts = 0
        self.edidble_prob = 0.5
        self.poisonous_prob = 0.5
        self.data = data
        self.edidble_features_likelihoods = {}
        self.poisonous_features_likelihoods = {}
        
    def loadData(self, data):
        self.data = data

    def compute_initial_probs(self):
        data = self.data
        nrecords = len(data)
        
        counts = data['class'].value_counts()
        edidble_counts = counts['e']
        poisonous_counts = counts['p']
        
        total = edidble_counts + poisonous_counts
        
        self.edidble_counts = edidble_counts
        self.poisonous_counts = poisonous_counts
        
        self.edidble_prob = edidble_counts / total
        self.poisonous_prob = poisonous_counts / total
    
        
        return self.edidble_prob, self.poisonous_prob
    
    def compute_histograms(self):
        df = self.data
        
        edidble_features_freq = defaultdict(dict)
        poisonous_features_freq = defaultdict(dict)
        total_edible_observations = self.edidble_counts * 22
        total_poisonous_observations = self.poisonous_counts * 22
        
        for _, row in df.iterrows():
            label = row['class']
            
            for feature in df.columns:
                if feature == 'class': continue
                
                value = row[feature]
                if label == 'e':
                    edidble_features_freq[feature][value] = edidble_features_freq[feature].get(value, 0) + 1
                else:
                    poisonous_features_freq[feature][value] = poisonous_features_freq[feature].get(value, 0) + 1
                    
        return edidble_features_freq, poisonous_features_freq, total_edible_observations, total_poisonous_observations
    

    def compute_likelihoods(self): 
        edidble_features_freq, poisonous_features_freq, total_edible_observations, total_poisonous_observations = self.compute_histograms()
        
        edidble_features_likelihoods = defaultdict(dict)
        poisonous_features_likelihoods = defaultdict(dict)
        
        for feature in edidble_features_freq:
            for value, freq in edidble_features_freq[feature].items():
                edidble_features_likelihoods[feature][value] = freq / total_edible_observations
        
        for feature in poisonous_features_freq:
            for value, freq in poisonous_features_freq[feature].items():
                poisonous_features_likelihoods[feature][value] = freq / total_poisonous_observations
                
        self.edidble_features_likelihoods, self.poisonous_features_likelihoods = edidble_features_likelihoods, poisonous_features_likelihoods
                    
        return self.edidble_features_likelihoods, self.poisonous_features_likelihoods
    
    def fit(self):
        self.compute_initial_probs()
        return self.compute_likelihoods()
    
    def predict(self, row: pd.Series):
        edidble_pred_score = np.log(self.edidble_prob)
        poisonous_pred_score = np.log(self.poisonous_prob)
        
        edidble_features_likelihoods = self.edidble_features_likelihoods
        poisonous_features_likelihoods = self.poisonous_features_likelihoods
        
        for feature, value in row.items():
            if feature == 'class': continue
            edidble_pred_score += np.log(edidble_features_likelihoods[feature].get(value, 1e-10))
            poisonous_pred_score += np.log(poisonous_features_likelihoods[feature].get(value, 1e-10))
            

        if edidble_pred_score > poisonous_pred_score:
            return 'e'
        else:
            return 'p'
            
            
    def test(self, test_data: pd.DataFrame) -> list:
        labels = []
        preds = []
        for _, row in test_data.iterrows():
            label = row['class']
            pred = self.predict(row=row)
            
            labels.append(label)
            preds.append(pred)
        return labels, preds
            
    def evaluate(self, test_results: list):
        labels = test_results[0]
        preds = test_results[1]

        print('===== accuracy =====')
        print(accuracy_score(labels, preds))
        print('====================\n')
        
        print('===== classification report =====')
        print(classification_report(labels, preds))
        print('=================================\n')
        
        print('===== confusion matrix =====')
        print(confusion_matrix(labels, preds))
        print('============================\n')    

In [5]:
df = pd.read_csv('data/mushrooms.csv')
train_df, test_df = train_test_split(df, test_size=0.2)
nb = NaiveBayesClassifier(data=train_df)
nb.fit()
test_results = nb.test(test_data=test_df)
nb.evaluate(test_results)

===== accuracy =====
0.9956923076923077

===== classification report =====
              precision    recall  f1-score   support

           e       1.00      0.99      1.00       851
           p       0.99      1.00      1.00       774

    accuracy                           1.00      1625
   macro avg       1.00      1.00      1.00      1625
weighted avg       1.00      1.00      1.00      1625


===== confusion matrix =====
[[845   6]
 [  1 773]]



In [7]:
encoded_df = encode_features(df)
pca = PCA(encoded_df)
pca_comps: np.ndarray = pca.compute_fprime()
print(construct_df(pca_comps, df['class']))

             0         1         2 class
0     0.228207 -0.345472  1.424425     p
1    -1.936561  4.796912  3.511668     e
2    -1.654173  2.464362  3.880827     e
3    -1.252026  1.679664  3.565488     p
4     1.581220 -1.002043  1.255760     e
...        ...       ...       ...   ...
8119 -6.469823 -2.308511 -0.588111     e
8120 -6.525909 -2.280695 -1.431399     e
8121 -1.860507  0.470244 -1.190212     e
8122  7.293068 -1.251693  1.241976     p
8123 -4.835533 -3.583758  0.112506     e

[8124 rows x 4 columns]
